# 3. Construcción de Agentes con LangChain

## Objetivos de Aprendizaje
- Comprender las ventajas de usar un framework como LangChain para el desarrollo de agentes.
- Aprender a definir herramientas (Tools) de forma sencilla con el decorador `@tool`.
- Crear un agente utilizando el constructor `create_openai_tools_agent`.
- Ejecutar el agente y gestionar la interacción mediante el `AgentExecutor`.

## ¿Por qué usar LangChain?

En los notebooks anteriores, construimos agentes desde cero. Primero, parseando texto, y luego, usando `function calling` nativo. Aunque el `function calling` es una gran mejora, todavía teníamos que:

1.  **Gestionar el historial de mensajes (`messages`) manualmente**: Añadir cada respuesta del usuario, del asistente y de la herramienta a la lista.
2.  **Orquestar el flujo de llamadas**: Escribir la lógica `if/else` para decidir si llamar a una herramienta, ejecutarla y volver a llamar al modelo.
3.  **Formatear las herramientas**: Escribir el JSON Schema para cada herramienta, lo cual es propenso a errores.

**LangChain** es un framework que abstrae toda esta complejidad. Actúa como una capa intermedia que simplifica enormemente la creación de aplicaciones basadas en LLMs, incluyendo los agentes. 

**Ventajas clave:**
- **Componentes Modulares**: Ofrece piezas reutilizables (LLMs, Prompts, Herramientas, etc.) que se pueden ensamblar fácilmente.
- **Agentes Listos para Usar**: Proporciona abstracciones de alto nivel como el `AgentExecutor` que manejan el ciclo de razonamiento (ReAct) por nosotros.
- **Integraciones**: Se conecta con cientos de fuentes de datos, APIs y otros servicios de forma nativa.

### 1. Instalación y Configuración

In [1]:
!pip install langchain langchain-openai openai wikipedia -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang('es')

# --- Configuración del LLM con LangChain ---
# LangChain actúa como un "envoltorio" (wrapper) sobre el cliente de OpenAI
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

c:\Users\realm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LLM de LangChain configurado.


### 2. Definición de Herramientas con LangChain

LangChain simplifica enormemente la creación de herramientas. En lugar de escribir un JSON Schema manual, simplemente usamos el decorador `@tool` sobre una función de Python. LangChain se encarga de inferir el esquema a partir de la firma de la función y su docstring.

El docstring es **muy importante**, ya que se usa como la descripción que el LLM ve para decidir si usar la herramienta o no.

In [3]:
from langchain_classic.tools import tool

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Es ideal para obtener información sobre personas, lugares o conceptos históricos y científicos."""
    try:
        wikipedia.set_lang("es")
        summary = wikipedia.summary(query, sentences=2)
        return summary
    except wikipedia.exceptions.PageError:
        return f"No se encontró ninguna página para '{query}'."
    except wikipedia.exceptions.DisambiguationError as e:
        return f"La búsqueda para '{query}' es ambigua. Opciones: {e.options[:3]}"

tools = [get_wikipedia_summary]

print("✅ Herramientas de LangChain definidas.")

✅ Herramientas de LangChain definidas.


### 3. Creación del Agente

Para crear el agente, necesitamos dos cosas:

1.  **Un Prompt**: Una plantilla que le dice al agente cómo razonar y cómo usar las herramientas. LangChain ya tiene prompts pre-construidos y optimizados para esto. Usaremos `hub.pull("hwchase17/openai-functions-agent")` para obtener una plantilla probada.
2.  **El Agente en sí**: Usamos la función `create_openai_tools_agent`, que une el LLM, las herramientas y el prompt.

El resultado es un `Runnable` de LangChain, que es el agente listo para ser ejecutado.

In [4]:
from langchain_classic import hub
from langchain_classic.agents import create_openai_tools_agent

# Descargar un prompt pre-diseñado y optimizado para agentes con function calling
prompt = hub.pull("hwchase17/openai-tools-agent")

# Unir el LLM, las herramientas y el prompt para crear el agente
agent = create_openai_tools_agent(llm, tools, prompt)

print("✅ Agente de LangChain creado.")

C:\Users\realm\AppData\Local\Temp\ipykernel_15324\3943055630.py:5: LangChainDeprecationWarning: langchain_classic.hub.pull is deprecated. Use the LangSmith SDK instead.
  prompt = hub.pull("hwchase17/openai-tools-agent")


ValueError: Pulling a public prompt by owner/name is disabled by default because prompts may contain untrusted serialized LangChain objects. If you trust this prompt, set `dangerously_pull_public_prompt=True` to acknowledge the risk.

### 4. Ejecución del Agente con `AgentExecutor`

El `agent` que creamos en el paso anterior es solo el "cerebro". Sabe cómo razonar, pero no puede ejecutar el ciclo de `Pensamiento -> Acción -> Observación` por sí mismo.

Para eso, usamos el **`AgentExecutor`**. Esta clase toma el agente y las herramientas, y se encarga de toda la orquestación:

- Llama al agente con la entrada del usuario.
- Si el agente decide usar una herramienta, el `AgentExecutor` la ejecuta.
- Pasa el resultado de la herramienta de vuelta al agente.
- Repite el proceso hasta que el agente da una respuesta final.
- Gestiona el historial de la conversación (`chat_history`).

In [ ]:
from langchain_classic.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ AgentExecutor listo para funcionar.")

✅ AgentExecutor listo para funcionar.


### 5. Invocando al Agente

Ahora, simplemente llamamos al método `invoke` del `agent_executor` con la pregunta. El parámetro `chat_history` es opcional pero útil para conversaciones de seguimiento.

In [ ]:
query = "¿Quién fue condorito y cuáles fueron sus logros más importantes?"

response = agent_executor.invoke({
    "input": query,
    "chat_history": []
})

print(f"🏁 Respuesta Final del Agente: {response}")



> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'Condorito'}`


Condorito es una serie de historieta cómica chilena protagonizada por el personaje homónimo. Fue publicada por primera vez el 6 de agosto de 1949 y creada por el dibujante chileno René Ríos Boettiger, más conocido como Pepo.​ Con los años, se convirtió en una de las historietas más populares de América Latina.Condorito es una historieta cómica chilena creada por René Ríos Boettiger, conocido como Pepo, y publicada por primera vez el 6 de agosto de 1949. A lo largo del tiempo, se convirtió en una de las historietas más populares de América Latina, destacándose por su humor y su conexión con la cultura popular de la región.

> Finished chain.
🏁 Respuesta Final del Agente: {'input': '¿Quién fue condorito y cuáles fueron sus logros más importantes?', 'chat_history': [], 'output': 'Condorito es una historieta cómica chilena creada por René Ríos Boettiger, conocido como Pepo, y publicad

## Conclusiones

Como hemos visto, LangChain reduce drásticamente la cantidad de código repetitivo y la complejidad de construir un agente. Nos hemos podido centrar en:

1.  **Definir la lógica de la herramienta**: La función `get_wikipedia_summary`.
2.  **Ensamblar los componentes**: Unir el LLM, las herramientas y un prompt usando las abstracciones de LangChain.

El `AgentExecutor` se encargó de todo el ciclo de ejecución, el manejo de estado y la orquestación, que antes teníamos que programar manualmente.

En el siguiente notebook, exploraremos **CrewAI**, otro framework de alto nivel que se especializa en la creación de equipos de agentes que colaboran para resolver tareas aún más complejas.